# 13 — Freddie retraining comparison: Stage 1 feasibility audit

**Purpose:** Determine whether a January 2023 frozen-versus-retrained XGBoost
comparison is supportable under the documented information-availability proxies.
This first stage does **not** train, load, rescore or change either model.

Run from the repository root or `notebooks/` in the project virtual environment.
This notebook is standalone: Notebook 12 does not need to be open or run in this kernel.
It reads existing local outputs, verifies their consistency and writes aggregate
review tables to a new timestamped directory.

## Scope and research question

When earlier, observable cohorts show deterioration, does an updated XGBoost improve
reliability on separate, later loans? This stage establishes the proposed samples
and evidence before implementing that comparison.

- Preserve the 24-month composite credit-event target and 29 predictor definitions.
- Preserve the 2015–2017 frozen baseline and its preprocessing contract.
- Proposed decision month: **2023-01**, subject to this audit.
- Initial benchmark update: original development loans plus **whole** earlier monitoring
  cohorts with all eligible labels ready under the proxy by that month.
- Proposed evaluation: **2023-H1 and 2023-H2**, kept separate from update loans.
- Use the existing 2018-H1 descriptive reliability anchor. It is not an acceptable-performance standard.
- Keep all existing illustrative deterioration sizes. They do not authorize retraining.
- The 2018–2023 outcomes were already explored. Any later comparison remains
  exploratory, even with strict separation of update and evaluation loans.

Historical label readiness is an assumption-based reconstruction, not proof of
publication timing. This audit does not claim that future 2023 predictors were
available in January 2023. The proposed model would be fixed then and applied
as later loans become observable.


## Added Stage 1B: compare training-population maturity rules

The initial whole-cohort feasibility audit below is preserved. Stage 1B at the end
compares that benchmark with individual-loan selection after the **full** 24-month
window has elapsed. The comparison does not choose a policy or fit a model.
Run all cells in order; share the new `notebook_13_stage1b_policy_review.zip`.
Previously saved Stage 1 outputs describe the original policy only. The added cells
must be executed locally before any new-policy counts can be reported.


## 1. Configuration and input fingerprints

No new data download is needed. Inputs are the existing modeling dataset,
baseline manifest/contract/model files, Notebook 11 predictions and summaries,
cleaned annual originations, target-candidate audit files and Notebook 12's
saved reliability-change CSV.

If multiple candidate-audit or change-summary files exist, explicitly select
the reviewed file below. The notebook never silently chooses the newest run.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, zipfile
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from IPython.display import display

ROOT = (Path.cwd().parent if Path.cwd().name.lower() == 'notebooks' else Path.cwd()).resolve()
DATA = ROOT / 'data' / 'processed' / 'Freddie_Mac'
BASE = DATA / 'model_baselines_2015_2017'
OOT = BASE / 'oot_evaluation'
KEYS = ['SOURCE YEAR', 'LOAN IDENTIFIER']
TARGET = 'TARGET_COMPOSITE_CREDIT_EVENT'
DECISION = pd.Period('2023-01', freq='M')
EVALUATION_PERIODS = ['2023-H1', '2023-H2']
ANCHOR = '2018-H1'
YEARS = list(range(2015, 2024))
# Use an absolute path or a path relative to the repository root if needed.
CHANGE_CSV = None
CANDIDATE_FILES = {}  # Example: {2015: 'data/processed/.../performance_target_candidates_2015.parquet'}

def require(condition, message):
    if not condition:
        raise ValueError(message)

def sha256(path):
    with Path(path).open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()

def select_file(explicit, matches, label):
    if explicit is not None:
        path = Path(explicit)
        path = path if path.is_absolute() else ROOT / path
        require(path.is_file(), f'{label}: configured path does not exist: {path}')
        return path.resolve()
    matches = sorted(set(Path(p).resolve() for p in matches))
    require(len(matches) == 1,
            f'{label}: expected exactly one file, found {len(matches)}. '
            f'Set the explicit configuration to the reviewed file. Candidates: {matches}')
    return matches[0]

require((ROOT / 'src' / 'freddie_features.py').is_file(), 'Open the repository root or notebooks directory.')
paths = {
    'modeling': DATA / 'freddie_modeling_dataset_2015_2023_24m.parquet',
    'manifest': BASE / 'model_manifest.json',
    'contract': BASE / 'freddie_feature_contract.json',
    'predictions': OOT / 'oot_predictions_2018_2023.parquet',
    'summary': OOT / 'cohort_reliability_summary.csv',
    'feature_source': ROOT / 'src' / 'freddie_features.py',
}
paths.update({f'origination_{year}': DATA / f'origination_{year}_clean.parquet' for year in YEARS})
for label, path in paths.items():
    require(path.is_file(), f'Missing input ({label}): {path}')

paths['changes'] = select_file(
    CHANGE_CSV, (DATA / 'notebook_12_reliability_changes').rglob('half_year_reliability_change_uncertainty.csv'),
    'Notebook 12 change uncertainty')
for year in YEARS:
    paths[f'candidate_{year}'] = select_file(
        CANDIDATE_FILES.get(year), DATA.rglob(f'performance_target_candidates_{year}.parquet'),
        f'{year} target audit')

manifest = json.loads(paths['manifest'].read_text(encoding='utf-8'))
contract = json.loads(paths['contract'].read_text(encoding='utf-8'))
require(manifest['target'] == TARGET and manifest['horizon_months'] == 24, 'Target contract differs.')
require(manifest['development_years'] == [2015, 2016, 2017], 'Development years differ.')
require(len(manifest['predictors']) == len(set(manifest['predictors'])) == 29, 'Expected 29 unique predictors.')
require(set(manifest['predictors']) == set(manifest['numeric_predictors']) | set(manifest['categorical_predictors']),
        'Manifest feature types do not cover the predictors.')
require(contract['training_years'] == [2015, 2016, 2017], 'Frozen preprocessing years differ.')
source = paths['feature_source'].read_bytes().replace(b'\r\n', b'\n')
source_blob = hashlib.sha1(b'blob ' + str(len(source)).encode() + b'\0' + source).hexdigest()
require(source_blob == '7da952013fd70095b001483f94256b4cf0a55e7d',
        'Feature source differs from the version reviewed in Notebook 12. Reconcile before continuing.')
for family in ['xgboost', 'logistic']:
    artifact = Path(manifest['models'][family]['artifact'])
    require(artifact.name == str(artifact), 'Expected a model filename within the baseline directory.')
    paths[f'model_{family}'] = BASE / artifact
    require(paths[f'model_{family}'].is_file(), f'Missing frozen {family} artifact.')
input_hashes = {label: sha256(path) for label, path in paths.items()}
require(input_hashes['modeling'] == manifest['input_file_sha256'], 'Modeling data changed since fitting.')
require(input_hashes['contract'] == manifest['feature_contract_sha256'], 'Frozen preprocessing changed.')
for family in ['xgboost', 'logistic']:
    require(input_hashes[f'model_{family}'] == manifest['models'][family]['sha256'], f'{family} artifact changed.')
print('Input fingerprints verified. No model was loaded, fitted or rescored.')


Input fingerprints verified. No model was loaded, fitted or rescored.


## 2. Reconcile loan keys, labels and saved reliability

The data are historical and already contain outcomes. Reading them for audit
does not make them permissible update data. The later selection step uses
cohort dates and readiness proxies, not event status or future performance.


In [3]:
def normalize(frame, label):
    frame = frame.copy()
    required = KEYS + ['ORIGINATION YEAR', 'ORIGINATION QUARTER', 'ORIGINATION COHORT']
    require(set(required).issubset(frame.columns), f'{label}: key/cohort columns missing.')
    require(not frame[required].isna().any().any(), f'{label}: missing key/cohort values.')
    for column in ['SOURCE YEAR', 'ORIGINATION YEAR']:
        values = pd.to_numeric(frame[column], errors='raise')
        require(np.isfinite(values).all() and values.eq(np.floor(values)).all(), f'{label}: invalid years.')
        frame[column] = values.astype(int)
    frame['LOAN IDENTIFIER'] = frame['LOAN IDENTIFIER'].astype('string')
    parts = frame['LOAN IDENTIFIER'].str.extract(r'^[FA](\d{2})Q([1-4])\d{7}$')
    require(parts.notna().all().all(), f'{label}: invalid identifier format.')
    q = pd.to_numeric(frame['ORIGINATION QUARTER'].astype('string').str.strip().str.removeprefix('Q'), errors='raise')
    require(q.isin([1, 2, 3, 4]).all(), f'{label}: invalid quarters.')
    require(frame['SOURCE YEAR'].isin(YEARS).all(), f'{label}: unsupported years.')
    require(frame['SOURCE YEAR'].eq(frame['ORIGINATION YEAR']).all(), f'{label}: source/origination year mismatch.')
    require(frame['SOURCE YEAR'].eq(2000 + parts[0].astype(int)).all() and q.eq(parts[1].astype(int)).all(),
            f'{label}: identifier/cohort mismatch.')
    require(frame['ORIGINATION COHORT'].astype('string').eq(frame['SOURCE YEAR'].astype(str) + 'Q' + q.astype(int).astype(str)).all(),
            f'{label}: cohort text mismatch.')
    frame['period'] = frame['SOURCE YEAR'].astype(str) + '-' + np.where(q.le(2), 'H1', 'H2')
    require(not frame.duplicated(KEYS).any(), f'{label}: duplicate keys.')
    return frame

meta = KEYS + ['ORIGINATION YEAR', 'ORIGINATION QUARTER', 'ORIGINATION COHORT']
modeling = normalize(pd.read_parquet(paths['modeling'], columns=meta + [TARGET, 'HORIZON_MONTHS']), 'modeling')
require(len(modeling) == 437668 and modeling['HORIZON_MONTHS'].eq(24).all(), 'Modeling totals or horizon differ.')
require(modeling[TARGET].isin([0, 1]).all(), 'Invalid target.')
development = modeling.loc[modeling['SOURCE YEAR'].le(2017)]
require(len(development) == manifest['development_rows'] == 143598, 'Development count differs.')
require(int(development[TARGET].sum()) == manifest['development_events'], 'Development events differ.')

pred = normalize(pd.read_parquet(paths['predictions']), 'saved predictions')
require(len(pred) == 294070 and int(pred['y_true'].sum()) == 6481, 'OOT totals differ.')
require(pred['SOURCE YEAR'].between(2018, 2023).all(), 'Unexpected prediction years.')
require(pred['y_true'].isin([0, 1]).all(), 'Invalid saved labels.')
for family in ['xgboost', 'logistic']:
    require(np.isfinite(pred[f'p_{family}']).all() and pred[f'p_{family}'].between(0, 1).all(),
            f'Invalid saved probabilities: {family}')
joined = modeling.loc[modeling['SOURCE YEAR'].ge(2018), KEYS + ['period', TARGET]].merge(
    pred[KEYS + ['period', 'y_true']], on=KEYS, how='outer', suffixes=('_data', '_pred'),
    validate='one_to_one', indicator=True)
require(joined['_merge'].eq('both').all() and joined['period_data'].eq(joined['period_pred']).all()
        and joined[TARGET].eq(joined['y_true']).all(), 'Prediction/data alignment differs.')

summary = pd.read_csv(paths['summary'], dtype={'period': 'string'})
half_summary = summary.loc[summary['period_kind'].eq('half_year')].copy()
require(len(half_summary) == 24 and not half_summary.duplicated(['period', 'model']).any(), 'Reliability grid differs.')
require(set(half_summary['model']) == {'xgboost', 'logistic'}, 'Model coverage differs.')
for (period, family), saved in half_summary.set_index(['period', 'model']).iterrows():
    part = pred.loc[pred['period'].eq(period)]
    y = part['y_true'].to_numpy(dtype=int)
    p = part[f'p_{family}'].to_numpy(dtype=float)
    require(len(part) > 0 and np.unique(y).size == 2, f'{period}: insufficient classes.')
    require(len(part) == int(saved['n']) and int(y.sum()) == int(saved['positives']), 'Reliability counts differ.')
    actual = [roc_auc_score(y, p), average_precision_score(y, p), brier_score_loss(y, p), y.mean()-p.mean(), y.mean()]
    np.testing.assert_allclose(actual, saved[['roc_auc', 'average_precision', 'brier', 'risk_gap', 'prevalence']].astype(float),
                               rtol=1e-9, atol=1e-12)
print('Loan keys, labels, counts and saved reliability reconcile. No predictions were regenerated.')


Loan keys, labels, counts and saved reliability reconcile. No predictions were regenerated.


## 3. Reconstruct the existing readiness proxy

Reuse Notebook 12's convention: first-payment month plus the saved observation
offset gives a reporting-month proxy. Nonterminated eligible loans wait through
month 23. For terminated loans, use the last observed month in the target window.
Treat that completion reporting month as usable from the following month.

This preserves the existing target and population. It does not reconstruct
archived releases or prove that target eligibility was known historically.


In [4]:
origin_parts, candidate_parts = [], []
for year in YEARS:
    part = pd.read_parquet(paths[f'origination_{year}'], columns=[c for c in meta if c != 'SOURCE YEAR'] + ['FIRST PAYMENT DATE'])
    part['SOURCE YEAR'] = year
    part = normalize(part, f'{year} originations')
    require(len(part) == 50000, f'{year}: expected 50,000 sampled originations.')
    payment = part['FIRST PAYMENT DATE'].astype('string').str.strip()
    require(payment.str.fullmatch(r'\d{6}', na=False).all(), f'{year}: invalid FIRST PAYMENT DATE.')
    part['first_payment_month'] = pd.to_datetime(payment, format='%Y%m', errors='raise').dt.to_period('M')
    origin_parts.append(part[KEYS + ['period', 'first_payment_month']])

    audit = pd.read_parquet(paths[f'candidate_{year}'], columns=[
        'LOAN IDENTIFIER', 'HORIZON_MONTHS', 'FIRST_OBSERVED_MONTH',
        'LAST_MONTH_WITHIN_HORIZON', 'TARGET_ELIGIBLE', 'TERMINATION_WITHIN_HORIZON'])
    audit = audit.loc[audit['HORIZON_MONTHS'].eq(24)].copy()
    audit['SOURCE YEAR'] = year
    audit['LOAN IDENTIFIER'] = audit['LOAN IDENTIFIER'].astype('string')
    for col in ['TARGET_ELIGIBLE', 'TERMINATION_WITHIN_HORIZON']:
        require(audit[col].notna().all() and audit[col].isin([True, False, 0, 1]).all(), f'{year}: invalid {col}.')
        audit[col] = audit[col].astype(bool)
    candidate_parts.append(audit)

originations = pd.concat(origin_parts, ignore_index=True)
candidates = pd.concat(candidate_parts, ignore_index=True)
require(not originations.duplicated(KEYS).any() and not candidates.duplicated(KEYS).any(), 'Duplicate origin/audit keys.')
require(originations.groupby('period').size().eq(25000).all(), 'Half-year origin sample counts differ.')
availability = originations.merge(candidates, on=KEYS, how='outer', validate='one_to_one', indicator=True)
require(availability['_merge'].eq('both').all(), 'Originations and target-candidate sets differ.')
availability = availability.drop(columns='_merge')
audit_keys = pd.MultiIndex.from_frame(availability.loc[availability['TARGET_ELIGIBLE'], KEYS])
model_keys = pd.MultiIndex.from_frame(modeling[KEYS])
require(len(audit_keys) == len(model_keys) and audit_keys.isin(model_keys).all()
        and model_keys.isin(audit_keys).all(), 'Eligible population changed.')

def whole_month_offsets(values, label, allow_missing=False):
    x = pd.to_numeric(values, errors='raise')
    require((allow_missing or x.notna().all()) and np.isfinite(x.dropna()).all()
            and x.dropna().eq(np.floor(x.dropna())).all(), f'Invalid {label} month offsets.')
    return x

first = whole_month_offsets(availability['FIRST_OBSERVED_MONTH'], 'first-observed', allow_missing=True)
availability['first_report_proxy'] = pd.PeriodIndex([
    payment + int(offset) if pd.notna(offset) else pd.NaT
    for payment, offset in zip(availability['first_payment_month'], first)], freq='M')
eligible = availability.loc[availability['TARGET_ELIGIBLE']].copy()
require(eligible['first_report_proxy'].notna().all(), 'Eligible reporting proxy missing.')
last = whole_month_offsets(eligible['LAST_MONTH_WITHIN_HORIZON'], 'last-within-window')
require(last.between(0, 23).all(), 'Last observation outside fixed target window.')
completion = np.where(eligible['TERMINATION_WITHIN_HORIZON'], last.to_numpy(), 23).astype(int)
eligible['completion_proxy'] = pd.PeriodIndex([
    payment + int(offset) for payment, offset in zip(eligible['first_payment_month'], completion)], freq='M')
require(eligible['first_report_proxy'].le(eligible['completion_proxy']).all(), 'Reporting begins after completion proxy.')
eligible['label_ready_proxy'] = eligible['completion_proxy'] + 1
eligible = eligible.merge(modeling[KEYS + [TARGET]], on=KEYS, validate='one_to_one')
model_ready_proxy = eligible.loc[eligible['SOURCE YEAR'].le(2017), 'label_ready_proxy'].max()
cohort_ready = eligible.groupby('period')['label_ready_proxy'].max()
require(len(cohort_ready) == 18, 'Expected 18 half-year cohorts across 2015–2023.')
print('Frozen baseline label-readiness proxy:', model_ready_proxy)
print('Proposed decision month:', DECISION)


Frozen baseline label-readiness proxy: 2022-10
Proposed decision month: 2023-01


## 4. Select proposed samples using dates, not outcomes

Use whole earlier monitoring cohorts only when every eligible loan's label meets
the readiness proxy. This avoids selecting early positive events while excluding
unresolved negative loans. It is conservative and may leave recent mature loans unused.

The 2023 event counts below describe retrospective evaluation support. They must
not determine the decision date, update sample, model settings or retraining trigger.


In [5]:
def period_start(period):
    return pd.Period(f"{period[:4]}-{'01' if period.endswith('H1') else '07'}", freq='M')

history_periods = sorted(p for p in cohort_ready.index if int(p[:4]) >= 2018
                         and period_start(p) < DECISION and cohort_ready.loc[p] <= DECISION)
require(not set(history_periods).intersection(EVALUATION_PERIODS), 'History/evaluation cohorts overlap.')
require(all(period_start(p) >= DECISION for p in EVALUATION_PERIODS), 'Evaluation starts before decision month.')
original_mask = eligible['SOURCE YEAR'].le(2017)
update_mask = original_mask | eligible['period'].isin(history_periods)
evaluation_mask = eligible['period'].isin(EVALUATION_PERIODS)
require(not (update_mask & evaluation_mask).any(), 'Update/evaluation masks overlap.')
update_keys = pd.MultiIndex.from_frame(eligible.loc[update_mask, KEYS])
evaluation_keys = pd.MultiIndex.from_frame(eligible.loc[evaluation_mask, KEYS])
require(not update_keys.isin(evaluation_keys).any(), 'Loan keys overlap.')
require(not eligible.loc[update_mask, 'LOAN IDENTIFIER'].isin(
    eligible.loc[evaluation_mask, 'LOAN IDENTIFIER']).any(), 'Loan identifiers overlap across source years.')
require(pd.MultiIndex.from_frame(development[KEYS]).isin(update_keys).all(), 'Original development loans omitted.')

audit_rows = []
for period, part in eligible.groupby('period', sort=True):
    year = int(period[:4])
    role = ('original_development' if year <= 2017 else
            'candidate_update_addition' if period in history_periods else
            'later_evaluation' if period in EVALUATION_PERIODS else 'not_selected')
    audit_rows.append({
        'period': period, 'role': role, 'eligible_loans': len(part),
        'events': int(part[TARGET].sum()), 'non_events': int(part[TARGET].eq(0).sum()),
        'prevalence': float(part[TARGET].mean()),
        'cohort_label_ready_proxy': str(cohort_ready.loc[period]),
        'labels_ready_by_decision': int(part['label_ready_proxy'].le(DECISION).sum()),
        'labels_not_ready_by_decision': int(part['label_ready_proxy'].gt(DECISION).sum()),
        'whole_cohort_labels_ready': bool(cohort_ready.loc[period] <= DECISION),
        'first_reports_not_usable_by_decision': int(part['first_report_proxy'].ge(DECISION).sum()),
        'counts_are_retrospective': True,
    })
cohort_audit = pd.DataFrame(audit_rows)
proposal_rows = []
for label, mask in [('original_development', original_mask),
                    ('additional_mature_history', eligible['period'].isin(history_periods)),
                    ('candidate_expanded_update', update_mask), ('later_evaluation', evaluation_mask)]:
    part = eligible.loc[mask]
    proposal_rows.append({'sample': label, 'loans': len(part), 'events': int(part[TARGET].sum()),
                          'non_events': int(part[TARGET].eq(0).sum()),
                          'periods': ', '.join(sorted(part['period'].unique())),
                          'has_both_classes': bool(part[TARGET].nunique() == 2)})
sample_proposal = pd.DataFrame(proposal_rows)
print('Fully mature monitoring cohorts at decision:', history_periods)
display(cohort_audit)
display(sample_proposal)
print('Counts do not establish sufficient precision or practical benefit. Review event support.')


Fully mature monitoring cohorts at decision: ['2018-H1', '2018-H2']


,period,role,eligible_loans,events,non_events,prevalence,cohort_label_ready_proxy,labels_ready_by_decision,labels_not_ready_by_decision,whole_cohort_labels_ready,first_reports_not_usable_by_decision,counts_are_retrospective
0,2015-H1,original_development,23602,87,23515,0.003686,2018-09,23602,0,True,0,True
1,2015-H2,original_development,23642,161,23481,0.006810,2019-05,23642,0,True,0,True
2,2016-H1,original_development,24039,157,23882,0.006531,2022-10,24039,0,True,0,True
3,2016-H2,original_development,24037,167,23870,0.006948,2020-07,24037,0,True,0,True
4,2017-H1,original_development,24091,222,23869,0.009215,2022-05,24091,0,True,0,True
5,2017-H2,original_development,24187,169,24018,0.006987,2022-03,24187,0,True,0,True
6,2018-H1,candidate_update_addition,23699,327,23372,0.013798,2021-11,23699,0,True,0,True
7,2018-H2,candidate_update_addition,23718,895,22823,0.037735,2022-12,23718,0,True,0,True
8,2019-H1,not_selected,24326,1106,23220,0.045466,2023-03,24324,2,False,0,True
9,2019-H2,not_selected,24285,1121,23164,0.046160,2023-06,24283,2,False,0,True


,sample,loans,events,non_events,periods,has_both_classes
0,original_development,143598,963,142635,"2015-H1, 2015-H2, 2016-H1, 2016-H2, 2017-H1, 2...",True
1,additional_mature_history,47417,1222,46195,"2018-H1, 2018-H2",True
2,candidate_expanded_update,191015,2185,188830,"2015-H1, 2015-H2, 2016-H1, 2016-H2, 2017-H1, 2...",True
3,later_evaluation,49501,876,48625,"2023-H1, 2023-H2",True


Counts do not establish sufficient precision or practical benefit. Review event support.


## 5. Review deterioration evidence available under the scenario

Only fully mature monitoring cohorts enter this decision-evidence table. Do not
use 2023 reliability to decide whether to retrain in January 2023.

Reuse Notebook 12's saved intervals, without rerunning bootstrap. Verify point
estimates and counts against Notebook 11. Preserve all four metrics and all
illustrative deterioration sizes. A supported statistical direction does not
establish practical importance.


In [6]:
METRICS = ['roc_auc', 'average_precision', 'brier', 'absolute_risk_gap']
CANDIDATE_SIZES = {
    'roc_auc': [0.01, 0.03, 0.05],
    'average_precision': [0.005, 0.01, 0.02],
    'brier': [0.001, 0.0025, 0.005],
    'absolute_risk_gap': [0.0025, 0.005, 0.01],
}
changes = pd.read_csv(paths['changes'], dtype={'period':'string', 'reference_period':'string'})
required_change = {'period', 'model', 'metric', 'reference_period', 'reference_value', 'current_value',
                   'worsening_change', 'worsening_ci_low', 'worsening_ci_high', 'valid_draws',
                   'reference_n', 'reference_events', 'current_n', 'current_events', 'direction_interpretation'}
require(required_change.issubset(changes.columns), 'Notebook 12 change-summary columns missing.')
require(len(changes) == 88 and not changes.duplicated(['period', 'model', 'metric']).any(),
        'Expected 88 unique Notebook 12 changes.')
require(set(changes['model']) == {'xgboost', 'logistic'} and set(changes['metric']) == set(METRICS),
        'Change-summary model or metric coverage differs.')
require(changes['reference_period'].eq(ANCHOR).all(), 'Change anchor differs.')
require(set(changes['period']) == set(pred['period']) - {ANCHOR}, 'Change cohort coverage differs.')
for row in changes.itertuples(index=False):
    reference = half_summary.loc[half_summary['period'].eq(ANCHOR) & half_summary['model'].eq(row.model)]
    current = half_summary.loc[half_summary['period'].eq(row.period) & half_summary['model'].eq(row.model)]
    require(len(reference) == len(current) == 1, 'Missing reliability reference/current row.')
    reference, current = reference.iloc[0], current.iloc[0]
    column = 'risk_gap' if row.metric == 'absolute_risk_gap' else row.metric
    a, b = float(reference[column]), float(current[column])
    if row.metric == 'absolute_risk_gap':
        a, b = abs(a), abs(b)
    direction = -1 if row.metric in ['roc_auc', 'average_precision'] else 1
    np.testing.assert_allclose([row.reference_value, row.current_value, row.worsening_change],
                               [a, b, direction*(b-a)], rtol=1e-9, atol=1e-12)
    require(row.reference_n == reference['n'] and row.reference_events == reference['positives']
            and row.current_n == current['n'] and row.current_events == current['positives'],
            'Change-summary counts differ.')
    require(np.isfinite([row.worsening_ci_low, row.worsening_ci_high]).all()
            and row.worsening_ci_low <= row.worsening_ci_high and row.valid_draws == 1000,
            'Invalid or incomplete saved bootstrap interval.')
    expected = ('worsening_direction_supported' if row.worsening_ci_low > 0 else
                'improvement_direction_supported' if row.worsening_ci_high < 0 else 'direction_uncertain')
    require(row.direction_interpretation == expected, 'Saved direction label differs from interval.')

decision_evidence = changes.loc[changes['model'].eq('xgboost') & changes['period'].isin(history_periods)].copy()
require(ANCHOR in history_periods or decision_evidence.empty, 'Anchor labels unavailable at decision.')
decision_evidence['cohort_label_ready_proxy'] = decision_evidence['period'].map(cohort_ready.astype(str))
decision_evidence['decision_month'] = str(DECISION)
decision_evidence['available_under_proxy'] = True
sensitivity_rows = []
for row in decision_evidence.itertuples(index=False):
    for size in CANDIDATE_SIZES[row.metric]:
        sensitivity_rows.append({
            'period': row.period, 'metric': row.metric, 'candidate_worsening_size': size,
            'worsening_change': row.worsening_change,
            'point_exceeds_candidate': bool(row.worsening_change > size),
            'interval_entirely_above_candidate': bool(row.worsening_ci_low > size),
            'interval_crosses_candidate': bool(row.worsening_ci_low <= size <= row.worsening_ci_high),
            'business_tolerance_validated': False,
        })
sensitivity = pd.DataFrame(sensitivity_rows, columns=[
    'period', 'metric', 'candidate_worsening_size', 'worsening_change', 'point_exceeds_candidate',
    'interval_entirely_above_candidate', 'interval_crosses_candidate', 'business_tolerance_validated'])
display(decision_evidence[['period','metric','worsening_change','worsening_ci_low',
                           'worsening_ci_high','direction_interpretation']])
display(sensitivity)
print('Positive change means worse than 2018-H1. No practical deterioration or retraining rule is approved.')


,period,metric,worsening_change,worsening_ci_low,worsening_ci_high,direction_interpretation
0,2018-H2,roc_auc,0.033920,0.002139,0.065522,worsening_direction_supported
1,2018-H2,average_precision,-0.045019,-0.060755,-0.028174,improvement_direction_supported
2,2018-H2,brier,0.023003,0.020300,0.025667,worsening_direction_supported
3,2018-H2,absolute_risk_gap,0.022954,0.020202,0.025580,worsening_direction_supported


,period,metric,candidate_worsening_size,worsening_change,point_exceeds_candidate,interval_entirely_above_candidate,interval_crosses_candidate,business_tolerance_validated
0,2018-H2,roc_auc,0.0100,0.033920,True,False,True,False
1,2018-H2,roc_auc,0.0300,0.033920,True,False,True,False
2,2018-H2,roc_auc,0.0500,0.033920,False,False,True,False
3,2018-H2,average_precision,0.0050,-0.045019,False,False,False,False
4,2018-H2,average_precision,0.0100,-0.045019,False,False,False,False
5,2018-H2,average_precision,0.0200,-0.045019,False,False,False,False
6,2018-H2,brier,0.0010,0.023003,True,True,False,False
7,2018-H2,brier,0.0025,0.023003,True,True,False,False
8,2018-H2,brier,0.0050,0.023003,True,True,False,False
9,2018-H2,absolute_risk_gap,0.0025,0.022954,True,True,False,False


Positive change means worse than 2018-H1. No practical deterioration or retraining rule is approved.


## 6. Feasibility checks and aggregate review bundle

Structural checks are not research approval. In particular, class presence is
not a minimum-event guarantee, and a candidate deterioration size is not a
business tolerance. Stop after exporting the bundle for review.


In [7]:
checks = pd.DataFrame([
    {'check':'Frozen baseline labels ready under proxy', 'passed':bool(model_ready_proxy <= DECISION)},
    {'check':'All proposed update labels ready under proxy', 'passed':bool(eligible.loc[update_mask, 'label_ready_proxy'].le(DECISION).all())},
    {'check':'At least one additional mature monitoring cohort', 'passed':bool(history_periods)},
    {'check':'All proposed update first reports precede decision', 'passed':bool(eligible.loc[update_mask, 'first_report_proxy'].lt(DECISION).all())},
    {'check':'Update sample contains both classes', 'passed':bool(eligible.loc[update_mask, TARGET].nunique() == 2)},
    {'check':'Each later evaluation cohort contains both classes (retrospective support)',
     'passed':bool(all(eligible.loc[eligible['period'].eq(p), TARGET].nunique() == 2 for p in EVALUATION_PERIODS))},
    {'check':'Decision evidence contains a later mature comparison', 'passed':bool(len(decision_evidence) > 0)},
    {'check':'No update/evaluation loan overlap', 'passed':bool(not update_keys.isin(evaluation_keys).any())},
])
display(checks)
structural_status = 'structurally_feasible_under_proxies_review_required' if checks['passed'].all() else 'blocked_review_required'
print('Audit result:', structural_status)

require(all(sha256(path) == input_hashes[label] for label, path in paths.items()), 'An input changed during the audit.')
specification = {
    'stage':'Notebook 13 Stage 1 feasibility only', 'audit_status':structural_status,
    'decision_month':str(DECISION), 'model_label_ready_proxy':str(model_ready_proxy),
    'target':TARGET, 'horizon_months':24, 'predictors':manifest['predictors'],
    'original_development_years':[2015,2016,2017], 'additional_history_periods':history_periods,
    'evaluation_periods':EVALUATION_PERIODS, 'reliability_anchor':ANCHOR,
    'update_policy':'Original development plus all fully mature whole earlier monitoring cohorts',
    'candidate_deterioration_sizes':CANDIDATE_SIZES,
    'business_tolerances_validated':False, 'retraining_approved':False,
    'model_fitted':False, 'predictions_regenerated':False,
    'independent_untouched_validation':False, 'historical_publication_dates_verified':False,
    'next_design_to_review':{
        'xgboost_settings':'Keep baseline settings fixed for the initial update comparison',
        'preprocessing':'Keep original frozen preprocessing for the initial comparison',
        'evaluation':'Same separate later loans for frozen and updated models',
        'metrics':METRICS,
        'uncertainty':'Paired loan bootstrap on identical evaluation loans; implementation pending',
    },
    'limitations':[
        'Readiness and reporting are proxies, not verified release or deployment dates.',
        'Current-snapshot eligibility and termination status are retrospectively known.',
        'Previously explored 2018–2023 outcomes make the comparison exploratory.',
        'Whole-cohort maturity is conservative and can exclude more recent mature individual loans.',
        'AP and Brier need prevalence context; pointwise intervals omit shared shocks and fitting uncertainty.',
        'Structural feasibility does not establish practical deterioration, retraining benefit or cost saving.',
    ],
    'input_files':[{'label':label,'path':str(path.relative_to(ROOT)) if path.is_relative_to(ROOT) else str(path),
                    'sha256':input_hashes[label]} for label,path in paths.items()],
}
output = DATA / 'notebook_13_retraining_feasibility' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
output.mkdir(parents=True, exist_ok=False)
tables = {'cohort_availability_audit.csv':cohort_audit, 'proposed_samples.csv':sample_proposal,
          'available_xgboost_deterioration.csv':decision_evidence,
          'candidate_deterioration_sensitivity.csv':sensitivity, 'feasibility_checks.csv':checks}
for filename, frame in tables.items():
    frame.to_csv(output / filename, index=False)
    readback = pd.read_csv(output / filename)
    require(len(readback) == len(frame) and list(readback.columns) == list(frame.columns), 'Export shape differs.')
(output / 'feasibility_specification.json').write_text(json.dumps(specification, indent=2, allow_nan=False), encoding='utf-8')
review_zip = output / 'notebook_13_stage1_review.zip'
with zipfile.ZipFile(review_zip, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for filename in [*tables, 'feasibility_specification.json']:
        archive.write(output / filename, arcname=filename)
print('Share this aggregate review ZIP:', review_zip)
print('STOP. No models were trained, loaded, recalibrated, rescored or replaced.')
print('Review the proposed samples and deterioration evidence before extending this notebook.')


,check,passed
0,Frozen baseline labels ready under proxy,True
1,All proposed update labels ready under proxy,True
2,At least one additional mature monitoring cohort,True
3,All proposed update first reports precede deci...,True
4,Update sample contains both classes,True
5,Each later evaluation cohort contains both cla...,True
6,Decision evidence contains a later mature comp...,True
7,No update/evaluation loan overlap,True


Audit result: structurally_feasible_under_proxies_review_required
Share this aggregate review ZIP: E:\capstone_project\mortgage-model-reliability\data\processed\Freddie_Mac\notebook_13_retraining_feasibility\20261022T214353645766Z\notebook_13_stage1_review.zip
STOP. No models were trained, loaded, recalibrated, rescored or replaced.
Review the proposed samples and deterioration evidence before extending this notebook.


## Stage 1 review and next policy comparison

The preceding Stage 1 audit evaluates the original whole-cohort benchmark.
Its saved counts and checks remain a record of that policy; they do not approve fitting.
Continue to Stage 1B below to compare it with a full-window loan-level alternative.

Keep the existing XGBoost settings, original frozen preprocessing contract,
29 predictor definitions, target and later evaluation populations fixed while reviewing
these candidate training populations. No policy is selected by 2023 outcomes.
Do not add fitting until the population counts, recency, readiness assumptions,
decision evidence and evaluation criteria have been reviewed.


# Stage 1B — Compare two candidate training-population rules

**A — Existing whole-cohort benchmark:** retain the original development loans and
add earlier half-year cohorts only when all eligible labels pass the existing
termination-aware readiness proxy. This preserves Stage 1's policy and its results.

**B — Full-window individual-loan alternative:** retain the same development loans
and add individual earlier monitoring loans only after months 0–23 from their recorded
first-payment date have elapsed. Treat the completed window as usable from the next
month, and require the first-report proxy to precede the decision month. Wait for the
full window even when a loan terminated or experienced a credit event earlier.

Selection B reads dates, cohort fields and the predefined evaluation-cohort list;
it does not read event labels, predictions, termination status or model performance.
Both rules operate on the same existing retrospectively eligible population. That
population definition and the next-month readiness convention remain explicit
assumptions, not evidence of historical predictor or label publication dates.

The original development population is held fixed. Check separately that its full
windows and first-report proxies satisfy B's timing assumptions. Any failed structural
check blocks further fitting; neither rule is automatically preferred.

The two rules need not be nested in every dataset: early termination can make a whole
cohort's labels ready before its last full calendar window ends. Report both directions
of any sample difference rather than assuming B contains A.

**Decision evidence stays separate:** the saved full-cohort deterioration evidence
above remains restricted to fully mature cohorts under the existing proxy. Partially
mature newer cohorts can supply date-eligible update loans under B, but their full-cohort
reliability must not be used as decision evidence before all relevant labels are ready.
Event counts and class checks below are retrospective descriptions of the fixed samples,
not an outcome-based selection rule or validated sample-size requirement.


In [ ]:
def full_window_addition_mask(date_frame, decision_month, evaluation_periods):
    # Only these date/cohort fields are used. In particular, no target is accessed.
    required = ['SOURCE YEAR', 'period', 'first_payment_month', 'first_report_proxy']
    require(set(required).issubset(date_frame.columns), 'Missing maturity-selection dates.')
    require(date_frame[required].notna().all().all(), 'Missing maturity-selection dates.')
    full_window_ready = date_frame['first_payment_month'] + 24
    earlier_period = date_frame['period'].map(period_start).lt(decision_month)
    return (date_frame['SOURCE YEAR'].ge(2018) & earlier_period
            & ~date_frame['period'].isin(evaluation_periods)
            & full_window_ready.le(decision_month)
            & date_frame['first_report_proxy'].lt(decision_month))

full_window_ready_proxy = eligible['first_payment_month'] + 24
selection_dates = eligible[['SOURCE YEAR', 'period', 'first_payment_month', 'first_report_proxy']].copy()
full_window_additions = full_window_addition_mask(selection_dates, DECISION, EVALUATION_PERIODS)
policy_masks = {
    'A_whole_cohort': update_mask.copy(),
    'B_full_window_individual': original_mask | full_window_additions,
}
policy_additions = {name: mask & ~original_mask for name, mask in policy_masks.items()}
policy_summaries, cohort_policy_rows, recency_rows, policy_check_rows = [], [], [], []
key_fingerprints = {}
for name, mask in policy_masks.items():
    addition_mask = policy_additions[name]
    selected_keys = pd.MultiIndex.from_frame(eligible.loc[mask, KEYS])
    evaluation_keys_b = pd.MultiIndex.from_frame(eligible.loc[evaluation_mask, KEYS])
    selected_ids = eligible.loc[mask, 'LOAN IDENTIFIER']
    both_keys = bool(selected_keys.isin(evaluation_keys_b).any())
    both_ids = bool(selected_ids.isin(eligible.loc[evaluation_mask, 'LOAN IDENTIFIER']).any())
    for sample, sample_mask in [('original_development', original_mask),
                                ('additional_history', addition_mask),
                                ('combined_candidate_update', mask),
                                ('later_evaluation', evaluation_mask)]:
        part = eligible.loc[sample_mask]
        policy_summaries.append({'policy': name, 'sample': sample, 'loans': len(part),
            'events': int(part[TARGET].sum()), 'non_events': int(part[TARGET].eq(0).sum()),
            'prevalence': float(part[TARGET].mean()) if len(part) else None,
            'periods': ', '.join(sorted(part['period'].unique())),
            'has_both_classes': bool(part[TARGET].nunique() == 2),
            'counts_are_retrospective': True})
    for period, part in eligible.groupby('period', sort=True):
        selected = part.loc[mask.loc[part.index]]
        cohort_policy_rows.append({'policy': name, 'period': period,
            'eligible_loans': len(part), 'selected_update_loans': len(selected),
            'selected_update_events': int(selected[TARGET].sum()),
            'selected_update_non_events': int(selected[TARGET].eq(0).sum()),
            'selected_fraction_of_eligible': len(selected) / len(part),
            'labels_ready_under_original_proxy': int(part['label_ready_proxy'].le(DECISION).sum()),
            'full_windows_ready_under_proxy': int(full_window_ready_proxy.loc[part.index].le(DECISION).sum()),
            'is_evaluation_cohort': bool(period in EVALUATION_PERIODS)})
    additions = eligible.loc[addition_mask]
    payment_ordinals = additions['first_payment_month'].map(lambda month: month.ordinal)
    median_age = (float(np.median([DECISION.ordinal - int(v) for v in payment_ordinals]))
                  if len(additions) else None)
    recency_rows.append({'policy': name, 'additional_loans': len(additions),
        'additional_loans_2019_onward': int(additions['SOURCE YEAR'].ge(2019).sum()),
        'additional_loans_2020_onward': int(additions['SOURCE YEAR'].ge(2020).sum()),
        'latest_selected_origination_half_year': max(additions['period']) if len(additions) else '',
        'latest_selected_recorded_first_payment': str(additions['first_payment_month'].max()) if len(additions) else '',
        'median_months_from_first_payment_to_decision': median_age,
        'selected_loans_with_full_windows_not_ready': int(full_window_ready_proxy.loc[mask].gt(DECISION).sum()),
        'selected_loans_with_labels_not_ready': int(eligible.loc[mask, 'label_ready_proxy'].gt(DECISION).sum())})
    specific_timing = (bool(full_window_ready_proxy.loc[mask].le(DECISION).all())
                       if name == 'B_full_window_individual'
                       else bool(eligible.loc[mask, 'label_ready_proxy'].le(DECISION).all()))
    rows = [
        ('Original development population retained', bool(mask.loc[original_mask].all())),
        ('Frozen baseline label-readiness proxy satisfied', bool(model_ready_proxy <= DECISION)),
        ('All selected labels ready under original proxy', bool(eligible.loc[mask, 'label_ready_proxy'].le(DECISION).all())),
        ('Selected first-report proxies precede decision', bool(eligible.loc[mask, 'first_report_proxy'].lt(DECISION).all())),
        ('Policy-specific maturity rule satisfied', specific_timing),
        ('Additional history is present', bool(addition_mask.any())),
        ('Combined update has both classes (descriptive support)', bool(eligible.loc[mask, TARGET].nunique() == 2)),
        ('No update/evaluation key or identifier overlap', bool(not both_keys and not both_ids)),
        ('No evaluation cohort enters update selection', bool(not eligible.loc[mask, 'period'].isin(EVALUATION_PERIODS).any())),
        ('Evaluation periods begin no earlier than decision', bool(all(period_start(v) >= DECISION for v in EVALUATION_PERIODS))),
        ('A mature later comparison is available as decision evidence', bool(len(decision_evidence) > 0)),
        ('Both later evaluation cohorts have both classes (retrospective support)',
            bool(all(eligible.loc[eligible['period'].eq(v), TARGET].nunique() == 2 for v in EVALUATION_PERIODS))),
    ]
    for label, passed in rows:
        policy_check_rows.append({'policy': name, 'check': label, 'passed': passed})
    normalized_keys = eligible.loc[mask, KEYS].sort_values(KEYS).astype(str)
    key_text = '\n'.join(normalized_keys.agg('|'.join, axis=1))
    key_fingerprints[name] = hashlib.sha256(key_text.encode('utf-8')).hexdigest()

policy_comparison = pd.DataFrame(policy_summaries)
policy_cohort_selection = pd.DataFrame(cohort_policy_rows)
policy_recency = pd.DataFrame(recency_rows)
policy_checks = pd.DataFrame(policy_check_rows)
whole_additions = policy_additions['A_whole_cohort']
individual_additions = policy_additions['B_full_window_individual']
whole_only = whole_additions & ~individual_additions
individual_only = individual_additions & ~whole_additions
difference_rows = []
for period, part in eligible.groupby('period', sort=True):
    idx = part.index
    difference_rows.append({'period': period,
        'selected_by_both_addition_rules': int((whole_additions & individual_additions).loc[idx].sum()),
        'whole_cohort_only_additions': int(whole_only.loc[idx].sum()),
        'full_window_individual_only_additions': int(individual_only.loc[idx].sum()),
        'neither_addition_rule': int((~whole_additions & ~individual_additions).loc[idx].sum())})
policy_differences = pd.DataFrame(difference_rows)

# Reconcile A against the preserved Stage 1 output rather than replacing that result.
for old_name, new_name in [('original_development', 'original_development'),
                           ('additional_mature_history', 'additional_history'),
                           ('candidate_expanded_update', 'combined_candidate_update'),
                           ('later_evaluation', 'later_evaluation')]:
    old_row = sample_proposal.loc[sample_proposal['sample'].eq(old_name)].iloc[0]
    new_row = policy_comparison.loc[policy_comparison['policy'].eq('A_whole_cohort')
                & policy_comparison['sample'].eq(new_name)].iloc[0]
    require(int(old_row['loans']) == int(new_row['loans'])
            and int(old_row['events']) == int(new_row['events']), 'Original policy counts changed.')

display(policy_comparison)
display(policy_recency)
display(policy_cohort_selection)
display(policy_differences)
display(policy_checks)
policy_status = ('structural_checks_passed_population_review_required' if policy_checks['passed'].all()
                 else 'blocked_review_required')
print('Policy comparison status:', policy_status)
print('Selection B uses date/cohort columns only; labels are summarized after selection.')
print('Neither policy is selected or approved for fitting.')


## Export Stage 1B and stop for review

This new bundle is separate from the preserved original Stage 1 bundle. It contains
aggregate counts and fingerprints of selected loan keys, not a new modeling dataset.
The input fingerprints must remain unchanged. The original model, predictions,
target and feature files are neither rewritten nor regenerated.

Review recency alongside the cohort mix and event support. More training loans do
not establish better adaptation, an optimal window or improved reliability. The
comparison remains exploratory, with historically unverified publication timing
and retrospectively determined eligible membership. Neither the full 2019–2023
reliability results nor 2023 evaluation outcomes choose the training rule.


In [ ]:
require(all(sha256(path) == input_hashes[label] for label, path in paths.items()),
        'An input changed during the policy comparison.')
policy_specification = {
    'stage': 'Notebook 13 Stage 1B training-population comparison only',
    'decision_month': str(DECISION), 'audit_status': policy_status,
    'target': TARGET, 'horizon_months': 24,
    'original_development_years': [2015, 2016, 2017],
    'evaluation_periods': EVALUATION_PERIODS, 'reliability_anchor': ANCHOR,
    'policies': {
        'A_whole_cohort': 'Preserved Stage 1 rule: original development plus whole earlier monitoring cohorts with every eligible label ready under the termination-aware proxy.',
        'B_full_window_individual': 'Same development loans plus individual earlier monitoring loans with first-payment month + 24 no later than decision, and first-report proxy before decision; wait for the full window regardless of early event or termination.',
    },
    'loan_key_fingerprints_sha256': key_fingerprints,
    'individual_selection_fields': list(selection_dates.columns),
    'individual_selection_uses_targets_predictions_or_termination': False,
    'decision_evidence_periods': sorted(decision_evidence['period'].unique().tolist()),
    'decision_evidence_policy': 'Retain only the already available full-cohort evidence; partial newer cohorts are not added to this evidence.',
    'selected_training_policy': None, 'retraining_approved': False,
    'model_fitted': False, 'predictions_regenerated': False, 'existing_inputs_changed': False,
    'historical_publication_dates_verified': False, 'independent_untouched_validation': False,
    'baseline_hyperparameters_and_preprocessing': 'Preserve for the initial comparison; fitting remains pending.',
    'limitations': [
        'The next-month convention and reporting dates are proxies, not verified historical publication dates.',
        'Both samples come from the existing retrospectively eligible population.',
        'Previously explored evaluation outcomes prevent untouched warning-validation claims.',
        'The observed monthly-gap limitation in two development loans remains documented; this audit does not relabel or exclude them.',
        'Recency and structural feasibility do not establish improvement, adequate precision or a retraining benefit.',
    ],
    'input_files': specification['input_files'],
}
policy_output = DATA / 'notebook_13_retraining_policy_comparison' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
policy_output.mkdir(parents=True, exist_ok=False)
policy_tables = {
    'training_policy_comparison.csv': policy_comparison,
    'policy_recency_summary.csv': policy_recency,
    'policy_cohort_selection.csv': policy_cohort_selection,
    'policy_difference_cohorts.csv': policy_differences,
    'policy_feasibility_checks.csv': policy_checks,
}
for name, frame in policy_tables.items():
    frame.to_csv(policy_output / name, index=False)
    readback = pd.read_csv(policy_output / name)
    require(len(readback) == len(frame) and list(readback.columns) == list(frame.columns),
            'Policy export shape differs.')
(policy_output / 'maturity_policy_specification.json').write_text(
    json.dumps(policy_specification, indent=2, allow_nan=False), encoding='utf-8')
policy_review_zip = policy_output / 'notebook_13_stage1b_policy_review.zip'
with zipfile.ZipFile(policy_review_zip, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for name in [*policy_tables, 'maturity_policy_specification.json']:
        archive.write(policy_output / name, arcname=name)
with zipfile.ZipFile(policy_review_zip) as archive:
    require(archive.testzip() is None and len(archive.infolist()) == 6, 'Policy ZIP verification failed.')
print('Share this new aggregate review ZIP:', policy_review_zip)
print('STOP. No policy selected; no models loaded, fitted, recalibrated, rescored or replaced.')


## Review before fitting

1. Compare A and B's selected loan counts, cohort mix, recency and event support.
2. Confirm the fixed decision date, complete-window convention, reporting proxy and
   zero overlap with the same later evaluation loans.
3. Keep the mature-cohort deterioration evidence separate from newer update-loan
   availability. Review all four metrics and their existing tradeoffs.
4. Choose and document a training-population policy before fitting; do not choose by
   the already observed 2023 reliability results.
5. Specify the initial frozen-versus-updated XGBoost comparison with fixed model
   settings and original preprocessing, separately saved artifacts and paired evaluation.

This notebook still contains no fitting implementation. The preserved frozen baseline,
targets and 29 predictor definitions remain unchanged. Retraining benefit, business
cost savings and external Fannie Mae generalizability remain unevaluated here.
